# Locally weighted average smoothing

Builds Queen-contiguity spatial weights over the population grid and computes a local spatial-lag average of case counts per cell (a smoothed disease surface), for the original dataset and every masked variant, applying the same `K_MIN`/`PMIN` publication rule as `Descriptive-Mapping.ipynb`.

**Input:** the datasets listed in `DATASETS` below, plus the population grid.
**Output:** one shapefile per dataset plus a summary CSV in `outputs_smoothed_disease_maps/`.


In [ ]:
# Imports
import os
from pathlib import Path

import numpy as np
import pandas as pd
import geopandas as gpd
import libpysal
from libpysal.weights import Queen, lag_spatial


In [ ]:
#Paths
# Data directory. Override with the DATA_DIR environment variable to
# point at a different copy of the data without editing this file, e.g.
#   DATA_DIR=/path/to/data jupyter execute this_notebook.ipynb
# The data itself is not part of this repository -- see README.md.
DATA_DIR = Path(os.environ.get("DATA_DIR", "data"))

GRID_PATH = DATA_DIR / "population_grid" / "GHS_Hungary_pop_city-clipped2.shp"
GRID_ID_COL = "GID"
POP_COL = "VALUE"

OUT_DIR = Path("outputs_smoothed_disease_maps")
OUT_DIR.mkdir(parents=True, exist_ok=True)

#Parameter
K_MIN = 5
PMIN = 50

In [ ]:
#Input datasets
DATASETS = {
    "original_points": {
        "type": "points",
        "path": DATA_DIR / "original" / "2008_eov_illness_stand.shp",
    },
    "donut": {
        "type": "points",
        "path": DATA_DIR / "donut_masking" / "donut_points150_high900.shp",
    },
    "stat_masked": {
        "type": "points",
        "path": DATA_DIR / "statistical_masking" / "statmasked_points.shp",
    },
    "synthetic_sdv": {
        "type": "points",
        "path": DATA_DIR / "synthetic" / "synthetic_points.shp",
    },
    "areal_grid_release": {
        "type": "grid",
        "path": DATA_DIR / "area_aggregation" / "aggregated_points.shp",
        "count_col": "n_cases_pu",
    },
}

In [ ]:
#Count cases per population-grid cell

def counts_by_grid(spec, grid_base):
    data = gpd.read_file(spec["path"]).to_crs(grid_base.crs)

    if spec["type"] == "points":
        joined = gpd.sjoin(
            data[["geometry"]],
            grid_base[[GRID_ID_COL, "geometry"]],
            how="inner",
            predicate="within",
        )
        return joined.groupby(GRID_ID_COL).size().reset_index(name="Y")
    
    if spec["type"] == "grid":
        count_col = spec["count_col"]
        data["Y"] = pd.to_numeric(data[count_col], errors="coerce").fillna(0)

        if GRID_ID_COL in data.columns:
            return data.groupby(GRID_ID_COL, as_index=False)["Y"].sum()
        
        # If the grid ID column is not present, we need to spatially join to the base grid to get the IDs
        joined = gpd.sjoin(
            data[["Y", "geometry"]],
            grid_base[[GRID_ID_COL, "geometry"]],
            how="inner",
            predicate="intersects",
        )
        return joined.groupby(GRID_ID_COL, as_index=False)["Y"].sum()

    raise ValueError(f"Unknown dataset type: {spec['type']}")

In [ ]:
#Loading population grid and build Queen spatial window
grid = gpd.read_file(GRID_PATH)
grid[POP_COL] = pd.to_numeric(grid[POP_COL], errors="coerce").fillna(0)

#We will use this base grid for all datasets, so we can build the spatial weights once and reuse them
grid_base = grid[[GRID_ID_COL, POP_COL, "geometry"]].copy().reset_index(drop=True)

#Queen contiguity from grid polygons
w = Queen.from_dataframe(grid_base, use_index=True)

#Include focal cell in the local window
w = libpysal.weights.fill_diagonal(w, 1.0)

#Row-standardization makes lag_spatial return an average
w.transform = "r"

In [ ]:
#Creating smoothed disease-map layers
summary_rows = []

#Process each dataset, create smoothed disease map, and apply publication rule
for name, spec in DATASETS.items():
    print(f"Processing {name}")

    counts = counts_by_grid(spec, grid_base)

    #Merge counts with grid base to ensure all grid cells are included, filling missing counts with 0
    out = grid_base.merge(counts, on=GRID_ID_COL, how="left")
    out["Y"] = out["Y"].fillna(0).astype(int)

    #Local average smoothing
    out["Y_smooth"] = lag_spatial(w, out["Y"].to_numpy(dtype=float))
    out["Y_smooth"] = out["Y_smooth"].round(3)

    #Publication rule: publish if smoothed count >= K_MIN and population >= PMIN
    out["publish"] = ((out["Y"] >= K_MIN) & (out[POP_COL] >= PMIN)).astype(int)

    #Exporting one shp per dataset for QGIS mapping
    out.to_file(OUT_DIR / f"{name}_smoothed_disease_map.shp", driver="ESRI Shapefile")

    #Collecting summary statistics for this dataset
    summary_rows.append({
        "dataset": name,
        "n_grid_cells": len(out),
        "n_cases_in_grid": int(out["Y"].sum()),
        "n_publishable_cells": int(out["publish"].sum()),
        "min_Y_smooth": float(out["Y_smooth"].min(skipna=True)),
        "max_Y_smooth": float(out["Y_smooth"].max(skipna=True)),
    })

summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv(OUT_DIR / "summary_smoothed_disease_maps.csv", index=False)

display(summary_df)
print("Done.")